# Library Imports

In [159]:
from agents import Agent, Runner, trace, WebSearchTool, ModelSettings, function_tool
import os
import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import display, Markdown
from typing import Dict
from datetime import datetime
from markdown_pdf import MarkdownPdf, Section
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


# Load Environment

In [160]:
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print(f"OPENAI_API_KEY loaded successfully: {OPENAI_API_KEY[:5]}...")
else:
    print("Error: OPENAI_API_KEY not found in environment variables.")

print("Environment variables loaded successfully.")

OPENAI_API_KEY loaded successfully: sk-pr...
Environment variables loaded successfully.


# Research Assistant Agent

In [161]:
RA_INSTRUCTIONS = """You are a research assistant. When given a search term, perform a web search on it and return a brief summary of what you find.
Your summary should be 2–3 paragraphs and remain below 300 words, focusing on the key points and core findings. 
Prioritise density over polish—full sentences and perfect grammar aren't necessary. 
Your output will feed into a larger report being assembled by someone else, so strip out filler and surface only the substance. 
Return the summary alone, with no preamble, commentary, or closing remarks."""

research_assistant_agent = Agent(
    name="Research Assistant",
    instructions=RA_INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4.1-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

print("Research Assistant agent created successfully.")

Research Assistant agent created successfully.


# Structured outputs and a description of the fields

## Planner Agent

In [162]:
number_of_searches = 2


searchplanner_agent_instructions = f"""
You are a helpful research assistant. Given a user's query, your job is to plan the research by generating a set of targeted web searches that together will yield the information needed to answer it well. 
Produce exactly {number_of_searches} search terms, each one focused on a distinct angle or sub-topic of the query so the results complement rather than duplicate each other.
 Return only the list of search terms, with no explanations or extra commentary.
"""

# Pydantic to define the schema of the response (Structured outputs)

In [163]:
class WebSearchItem(BaseModel):
    query: str = Field(description="The search term to use.")
    reason: str = Field(description="A brief explanation of why this search is useful.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the user's query."
    )

searchplanner_agent = Agent(
    name="Search Planner",
    instructions=searchplanner_agent_instructions,
    model="gpt-4.1-mini",
    output_type=WebSearchPlan,
)

In [164]:
from pathlib import Path
print(Path.cwd())

/Users/yasinemirkutlu/agents/yek_openai2


In [165]:
from pathlib import Path
reports = Path.cwd() / "reports"
print("Folder exists:", reports.exists())
if reports.exists():
    for f in sorted(reports.iterdir()):
        print(f.name)

Folder exists: True
Latest_Advancements_in_Renewable_Energy_Technologies_and_Their_Impact_on_Global_Energy_Markets_in_the_Next_Decade_20260411_221406.pdf
Research_Report_on_Impact_of_Climate_Change_on_Marine_Biodiversity_20260411_211954.pdf


# PDF Creater Tool Function

In [166]:
@function_tool
def save_pdf_report(title: str, markdown_body: str) -> Dict[str, str]:
    """Save a research report as a PDF in a 'reports' folder inside the project directory."""
    project_dir = Path.cwd()
    reports_dir = project_dir / "reports"
    reports_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_title = "".join(c if c.isalnum() or c in " -_" else "" for c in title).strip().replace(" ", "_")
    filepath = reports_dir / f"{safe_title}_{timestamp}.pdf"

    # Replace fancy Unicode punctuation with plain ASCII equivalents
    replacements = {
        "\u2014": "-",   # em dash —
        "\u2013": "-",   # en dash –
        "\u2018": "'",   # left single quote '
        "\u2019": "'",   # right single quote '
        "\u201c": '"',   # left double quote "
        "\u201d": '"',   # right double quote "
        "\u2026": "...", # ellipsis …
        "\u00a0": " ",   # non-breaking space
        "\u2212": "-",   # minus sign −
        "\u2022": "*",   # bullet •
    }
    clean_body = markdown_body
    for fancy, plain in replacements.items():
        clean_body = clean_body.replace(fancy, plain)

    pdf = MarkdownPdf(toc_level=2)
    pdf.add_section(Section(clean_body))
    pdf.meta["title"] = title
    pdf.save(str(filepath))

    print(f"PDF saved to: {filepath.resolve()}")
    return {"status": "success", "filepath": str(filepath.resolve())}

In [167]:
save_pdf_report

FunctionTool(name='save_pdf_report', description="Save a research report as a PDF in a 'reports' folder inside the project directory.", params_json_schema={'properties': {'title': {'title': 'Title', 'type': 'string'}, 'markdown_body': {'title': 'Markdown Body', 'type': 'string'}}, 'required': ['title', 'markdown_body'], 'title': 'save_pdf_report_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x114187420>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

# PDF Creator Agent

In [168]:
pdf_agent_instructions = """
You save research reports as PDFs. You will receive a markdown report.
   You MUST call the save_pdf_report tool exactly once, passing:
   - title: a short descriptive title derived from the report's actual H1 heading
   - There must be only one H1 heading in the markdown, and it must be the first line. If there is no H1 heading, or if there are multiple, respond with an error instead of inventing a title.
   - markdown_body: the EXACT markdown you received, character-for-character, with no edits, no rewrites, no summarization, no substitution, no invented content
   Do NOT generate your own report. Do NOT use placeholder examples. If the input is empty, respond with an error instead of inventing content.
   Use only plain ASCII punctuation: straight quotes, hyphens, and standard dashes — no em dashes, en dashes, or curly quotes.
"""


pdf_agent = Agent(
    name="PDF Report Saver",
    instructions=pdf_agent_instructions,
    tools=[save_pdf_report],
    model="gpt-4.1-mini",
)

print("PDF Report Saver agent created successfully.")

PDF Report Saver agent created successfully.


# Writer Agent

In [169]:
writer_agent_instructions = """
You are a senior researcher responsible for producing a cohesive, well-structured report in response to a research query. 
You will receive the original query along with preliminary findings gathered by a research assistant.
Begin by drafting an outline that lays out the structure and narrative flow of the report. 
Once the outline is in place, write the full report based on it and return that as your final output.
The report must be in markdown format and should be substantial in both length and depth — 
Your must produce roughly 3–6 pages of content and the content should not be fewer than 1,000 words."""

#Pydantic models for the writer agent's output

class ReportOutline(BaseModel):
    summary: str = Field(description="A brief summary of the main findings and conclusions that the report will present.")

    report: str = Field(description="The full markdown body of the report, structured according to the outline and incorporating the research assistant's findings.")

    suggested_questions: list[str] = Field(description="A list of any additional questions that should be explored to deepen the report, based on gaps or interesting leads in the research assistant's findings.")

writer_agent = Agent(
    name="Report Writer",
    instructions=writer_agent_instructions,
    model="gpt-4.1-mini",
    output_type=ReportOutline,
)

print("Report Writer agent created successfully.")

Report Writer agent created successfully.


# Plan and Execute the search

In [170]:
async def search_planner(query: str):
    """Use the search planner agent to generate a web search plan for a given query."""
    print(f"Planning Searchers. Generating search plan for query: {query}")
    results = await Runner.run(searchplanner_agent, f"Query: {query}")
    #print(f"Will perform {len(results.final_output.searches)} searches")
    return results.final_output

async def do_search(search_plan: WebSearchPlan):
    """Use the research assistant agent to perform web searches based on a search plan."""
    #print(f"Performing Searches. Executing search plan: {search_plan.searches}")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print(f"Searches completed. Results: {results}")
    return results

async def search(item: WebSearchItem):
    """Perform a single web search using the research assistant agent."""
    print(f"Performing Search. Searching for: {item}")
    input = f"Search term: {item.query} and Search rationale: {item.reason}"
    result = await Runner.run(research_assistant_agent, input)
    print(f"Search completed for '{item}'. Result: {result}")
    return result.final_output


# Produce PDF 

In [171]:
async def produce_pdf_report(query: str, search_results: list[str]):
    """Use the writer agent to write a report, then save it as a PDF via the report agent."""
    print("Thinking about report...")
    writer_input = f"Original query: {query}\nSummarized search results: {search_results}"
    writer_result = await Runner.run(writer_agent, writer_input)
    report_data = writer_result.final_output  # ReportData instance
    print("Finished writing report")

    print("Saving report as PDF...")
    report_input = (
        "Please save the following research report as a PDF. "
        "Call the save_pdf_report tool with an appropriate title and the markdown body below.\n\n"
        f"{report_data.report}"
    )
    print("report_input length:", len(report_input))
    print("report_input end:", report_input[-300:])
    await Runner.run(pdf_agent, report_input)
    print("PDF saved")

    return report_data

# Execution

In [172]:
query = "What are the latest advancements in renewable energy technologies, and how might they impact global energy markets in the next decade?"

with trace("Deep Research Agentic AI"):
    print("Starting research process...")
    search_plan = await search_planner(query)
    search_results = await do_search(search_plan)
    report_data = await produce_pdf_report(query, search_results)
    print("Research process completed.")
display(Markdown(f"## Research Summary\n\n{report_data.summary}"))


Starting research process...
Planning Searchers. Generating search plan for query: What are the latest advancements in renewable energy technologies, and how might they impact global energy markets in the next decade?
Performing Search. Searching for: query='latest advancements in renewable energy technologies 2024' reason='To find the most recent innovations and technological developments in renewable energy.'
Performing Search. Searching for: query='impact of renewable energy advancements on global energy markets in next decade' reason='To understand how new renewable technologies might influence global energy markets over the next ten years.'
Search completed for 'query='latest advancements in renewable energy technologies 2024' reason='To find the most recent innovations and technological developments in renewable energy.''. Result: RunResult:
- Last agent: Agent(name="Research Assistant", ...)
- Final output (str):
    In 2024, the renewable energy sector experienced significant t

## Research Summary

This report examines the latest advancements in renewable energy technologies as of 2024 and analyzes their projected impacts on global energy markets over the next decade. Key developments include breakthroughs in solar and wind energy technologies, significant expansions in renewable energy capacity worldwide, and the integration of advanced energy storage solutions that enhance grid stability. The report further explores regional leadership in renewables, corporate adoption trends, and the broader implications for energy market dynamics, carbon emissions, and the transition to a sustainable, low-carbon global economy.